# Understanding Association Rule Metrics: Three Approaches

This notebook examines the rule `Neutrophil_CENTER → FibroticEpithelial_NEIGHBOR` in FOV **GVHD_32_FOV_1**  
using three different ways to encode spatial neighborhoods into transactions.  

Each section explains what **support**, **confidence**, **lift**, and **conviction** mean  
for this specific biological question: *are Neutrophils and FibroticEpithelial cells spatially co-located?*

In [64]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import sys
sys.path.append("..")
from utils.spatial import is_dominated

In [65]:
# Load cell table for the FOV of interest
cell_table = pd.read_csv("../data/MIBIGutCsv/cell_table.csv")
FOV = "GVHD_32_FOV_1"
fov_data = cell_table[cell_table["fov"] == FOV].copy().reset_index(drop=True)

# Spatial parameters — same as the weighted FP-Growth algorithm
SCALE     = 400.0 / 1024.0   # pixels → microns
RADIUS    = 25.0              # neighborhood radius (microns)
BANDWIDTH = 15.0              # Gaussian bandwidth; at distance=BANDWIDTH, weight ≈ 0.61

coords     = fov_data[["centroid_x", "centroid_y"]].values * SCALE
cell_types = fov_data["cell type"].values

print(f"FOV: {FOV}")
print(f"Total cells: {len(cell_types)}")
print()
print("Cell type counts:")
print(pd.Series(cell_types).value_counts().to_string())

FOV: GVHD_32_FOV_1
Total cells: 2579

Cell type counts:
Muscle                1005
Neutrophil             484
Macrophage             274
FibroticEpithelial     172
Fibroblast             141
Neutrophil_CD15        130
CD8T                   109
SMV                     99
Neuron                  51
Macrophage_Calp         27
Endothelial             21
Mesenchymal_VIM         16
CD4T                    13
ImmuneOther_CD45RA       7
Epithelial               6
Mast                     5
ImmuneOther              4
Treg                     4
CD3T                     3
Monocyte                 3
Plasma                   2
Plasma_CD38              2
Unidentified             1


In [66]:
# Find all cells within RADIUS for each cell (same as the algorithm)
nn = NearestNeighbors(radius=RADIUS).fit(coords)
neighbors_idx = nn.radius_neighbors(coords, return_distance=False)

avg_neighbors = np.mean([len(n) - 1 for n in neighbors_idx])  # -1 to exclude self
print(f"Neighborhoods built (radius={RADIUS} microns)")
print(f"Average neighbors per cell: {avg_neighbors:.1f}")

Neighborhoods built (radius=25.0 microns)
Average neighbors per cell: 32.9


In [67]:
def compute_metrics(transactions, A, B):
    """
    Compute association rule metrics for the rule A -> B.
    Works for all three transaction types (binary, normalized, capped).

    support(X)    = average weight of X across ALL transactions
    support(A, B) = average of min(w_A, w_B) across all transactions
                    (standard weighted extension of intersection support)
    confidence    = support(A, B) / support(A)
    lift          = confidence / support(B)
    conviction    = (1 - support(B)) / (1 - confidence)
    """
    N = len(transactions)
    supp_A  = sum(t.get(A, 0) for t in transactions) / N
    supp_B  = sum(t.get(B, 0) for t in transactions) / N
    supp_AB = sum(min(t[A], t[B]) for t in transactions if A in t and B in t) / N
    conf = supp_AB / supp_A if supp_A > 0 else 0
    lift = conf / supp_B    if supp_B > 0 else 0
    conv = (1 - supp_B) / (1 - conf) if conf < 1.0 else float("inf")
    return dict(N=N, supp_A=supp_A, supp_B=supp_B, supp_AB=supp_AB,
                confidence=conf, lift=lift, conviction=conv)


def print_metrics(m, A, B):
    print(f"Rule:  {A}")
    print(f"    ->  {B}")
    print(f"  support(A)   = {m['supp_A']:.4f}  (avg weight of A per transaction; binary: fraction of {m['N']} cells where A is CENTER)")
    print(f"  support(B)   = {m['supp_B']:.4f}  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)")
    print(f"  support(A,B) = {m['supp_AB']:.4f}  (avg min(w_A, w_B) per transaction; binary: fraction where both co-occur)")
    print(f"  confidence   = {m['confidence']:.3f}   (support(A,B) / support(A); binary: P(B present | A is CENTER) = {m['confidence']*100:.1f}%)")
    print(f"  lift         = {m['lift']:.2f}   (confidence / support(B); >1 means co-occurrence stronger than chance)")
    print(f"  conviction   = {m['conviction']:.2f}  (how often A occurs without B being incorrectly predicted absent; >1 means rule is informative)")

---
## Section 1 — Binary Approach (Presence / Absence)

### What is it?
Each neighbor cell type is simply **present (1) or absent (0)** in a cell's neighborhood.  
No distance weighting — a cell at 2 microns and a cell at 24 microns count equally.

---

### Metric Definitions (for this biological case)

**support(Neutrophil_CENTER)**  
= fraction of all cells in the FOV that are Neutrophils  
*"How abundant are Neutrophils in this tissue?"*

**support(FibroticEpithelial_NEIGHBOR)**  
= fraction of all cells that have **at least one** FibroticEpithelial neighbor within 25 microns  
*"What fraction of cells live near a Fibrotic cell?"* — this is the **baseline rate**

**support(A, B)**  
= fraction of all cells that are Neutrophils AND have at least one Fibrotic neighbor  
*"What is the joint probability of seeing this pair?"*

**confidence(A -> B)** = support(A,B) / support(A)  
= P(Fibrotic neighbor | Neutrophil center)  
*"Of all Neutrophil cells, what fraction have a Fibrotic neighbor?"*  
Ranges 0-1. High confidence means B almost always appears in A's neighborhood.

**lift(A -> B)** = confidence / support(B)  
*"Are Neutrophils more likely than a random cell to have a Fibrotic neighbor?"*  
lift = 1.0: no association (random). lift > 1: positive association. lift < 1: avoidance.

**conviction(A -> B)** = (1 - support(B)) / (1 - confidence)  
*"How much stronger is the rule than pure chance?"*  
conviction = 1.0: independent. conviction -> infinity: confidence = 1 (B always appears with A).

In [68]:
# Build binary transactions
# Each item maps to 1 (present) — absence means item is not in the dict

transactions_binary = []

for center_i, idxs in enumerate(neighbors_idx):
    neighbor_idxs = [n for n in idxs if n != center_i]
    if len(neighbor_idxs) == 0:
        continue
    if is_dominated(cell_types[list(idxs)]):
        continue

    trans = {}
    for n_idx in neighbor_idxs:
        trans[f"{cell_types[n_idx]}_NEIGHBOR"] = 1  # presence only, no weight
    trans[f"{cell_types[center_i]}_CENTER"] = 1

    transactions_binary.append(trans)

print(f"Total transactions: {len(transactions_binary)}")
print(f"(One transaction per cell in the FOV)")

Total transactions: 1877
(One transaction per cell in the FOV)


In [69]:
# Inspect a sample Neutrophil-centered transaction
sample = next(t for t in transactions_binary if "Neutrophil_CENTER" in t)
print("Example Neutrophil-centered transaction (binary):")
for item in sorted(sample):
    print(f"  {item}: {sample[item]}")
print()
print("Every value is 1 (present). The transaction only records WHICH types appear.")

Example Neutrophil-centered transaction (binary):
  CD8T_NEIGHBOR: 1
  Fibroblast_NEIGHBOR: 1
  FibroticEpithelial_NEIGHBOR: 1
  Macrophage_NEIGHBOR: 1
  Muscle_NEIGHBOR: 1
  Neuron_NEIGHBOR: 1
  Neutrophil_CD15_NEIGHBOR: 1
  Neutrophil_CENTER: 1

Every value is 1 (present). The transaction only records WHICH types appear.


In [70]:
# Single-item supports = simple cell type frequencies
N = len(transactions_binary)

neut_center  = sum(1 for t in transactions_binary if "Neutrophil_CENTER"           in t)
neut_neigh   = sum(1 for t in transactions_binary if "Neutrophil_NEIGHBOR"         in t)
fibro_center = sum(1 for t in transactions_binary if "FibroticEpithelial_CENTER"   in t)
fibro_neigh  = sum(1 for t in transactions_binary if "FibroticEpithelial_NEIGHBOR" in t)

print(f"support(Neutrophil_CENTER)           = {neut_center/N:.3f}  ({neut_center}/{N} cells)")
print(f"support(Neutrophil_NEIGHBOR)         = {neut_neigh/N:.3f}  ({neut_neigh}/{N} cells)")
print(f"support(FibroticEpithelial_CENTER)   = {fibro_center/N:.3f}  ({fibro_center}/{N} cells)")
print(f"support(FibroticEpithelial_NEIGHBOR) = {fibro_neigh/N:.3f}  ({fibro_neigh}/{N} cells have >= 1 Fibrotic neighbor)")
print()
print("-> support(CENTER) = true cell type frequency (no distortion)")
print(f"-> {fibro_neigh/N*100:.0f}% of cells have a Fibrotic neighbor because 25 micron radius is large")

support(Neutrophil_CENTER)           = 0.255  (478/1877 cells)
support(Neutrophil_NEIGHBOR)         = 0.752  (1412/1877 cells)
support(FibroticEpithelial_CENTER)   = 0.092  (172/1877 cells)
support(FibroticEpithelial_NEIGHBOR) = 0.711  (1335/1877 cells have >= 1 Fibrotic neighbor)

-> support(CENTER) = true cell type frequency (no distortion)
-> 71% of cells have a Fibrotic neighbor because 25 micron radius is large


In [71]:
# Rule metrics: both directions of Neutrophil <-> FibroticEpithelial, and Macrophage self-association
m1 = compute_metrics(transactions_binary, "Neutrophil_CENTER",         "FibroticEpithelial_NEIGHBOR")
m2 = compute_metrics(transactions_binary, "FibroticEpithelial_CENTER", "Neutrophil_NEIGHBOR")
m3 = compute_metrics(transactions_binary, "Macrophage_CENTER",         "Macrophage_NEIGHBOR")

print("=" * 60)
print_metrics(m1, "Neutrophil_CENTER", "FibroticEpithelial_NEIGHBOR")
print()
print("=" * 60)
print_metrics(m2, "FibroticEpithelial_CENTER", "Neutrophil_NEIGHBOR")
print()
print("=" * 60)
print_metrics(m3, "Macrophage_CENTER", "Macrophage_NEIGHBOR")

Rule:  Neutrophil_CENTER
    ->  FibroticEpithelial_NEIGHBOR
  support(A)   = 0.2547  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 0.7112  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.2009  (avg min(w_A, w_B) per transaction; binary: fraction where both co-occur)
  confidence   = 0.789   (support(A,B) / support(A); binary: P(B present | A is CENTER) = 78.9%)
  lift         = 1.11   (confidence / support(B); >1 means co-occurrence stronger than chance)
  conviction   = 1.37  (how often A occurs without B being incorrectly predicted absent; >1 means rule is informative)

Rule:  FibroticEpithelial_CENTER
    ->  Neutrophil_NEIGHBOR
  support(A)   = 0.0916  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 0.7523  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.0858  (avg min(w_A, w_B) per transac

In [72]:
# Interpret the results
print("INTERPRETATION")
print("-" * 60)
print(f"Neutrophil -> Fibrotic: lift = {m1['lift']:.2f}")
print(f"  Neutrophil cells are {m1['lift']:.2f}x more likely to have a Fibrotic neighbor")
print(f"  than a random cell in this FOV.")
print(f"  confidence = {m1['confidence']*100:.0f}%: {m1['confidence']*100:.0f}% of Neutrophil cells have >= 1 Fibrotic neighbor")
print()
print(f"Fibrotic -> Neutrophil: lift = {m2['lift']:.2f}")
print(f"  confidence = {m2['confidence']*100:.0f}%: {m2['confidence']*100:.0f}% of Fibrotic cells have >= 1 Neutrophil neighbor")
print()
print(f"Macrophage -> Macrophage: lift = {m3['lift']:.2f}")
print()
print("NOTE: Binary lift values close to 1 mean cells are barely more co-located than random.")
print("The 25 micron radius is large — nearly every cell happens to have most cell types as neighbors.")

INTERPRETATION
------------------------------------------------------------
Neutrophil -> Fibrotic: lift = 1.11
  Neutrophil cells are 1.11x more likely to have a Fibrotic neighbor
  than a random cell in this FOV.
  confidence = 79%: 79% of Neutrophil cells have >= 1 Fibrotic neighbor

Fibrotic -> Neutrophil: lift = 1.24
  confidence = 94%: 94% of Fibrotic cells have >= 1 Neutrophil neighbor

Macrophage -> Macrophage: lift = 1.08

NOTE: Binary lift values close to 1 mean cells are barely more co-located than random.
The 25 micron radius is large — nearly every cell happens to have most cell types as neighbors.


---
## Section 2: Normalize All Weights to Sum = 1

### What is it?
Each neighbor is weighted by a **Gaussian decay**: `w(d) = exp(-d^2 / 2*sigma^2)`  
A close neighbor gets weight close to 1.0; a far neighbor gets smaller weight.  
The center cell gets weight = 1.0.  
Then **all** weights (center + all neighbors) are divided by their total — they sum to 1.

---

### Metric Definitions (for this biological case)

**support(FibroticEpithelial_CENTER)**  
= average normalized CENTER weight across Fibrotic-centered transactions  
Should equal cell type frequency (~0.09), but is much lower in this FOV.  
**Why?** FibroticEpithelial cells are surrounded by many Neutrophils -> large total weight ->  
CENTER shrinks to 1 / (large total).  
**This is the core bug**: support is too low because of normalization, not because Fibrotic cells are rare.

**support(A, B)**  
= average of min(normalized_w_A, normalized_w_B) across all transactions  
Since both weights are shrunk by the same large denominator, pair support is doubly diluted.

**confidence(A -> B)**  
= relative weight of B in A-centered neighborhoods.  
Can appear high even when absolute co-occurrence is weak.

**lift(A -> B)**  
**Artificially inflated** due to normalization:  
support(A) is shrunk by dense neighborhoods; support(B) is shrunk for the same reason.  
Their ratio (lift) gets amplified beyond the true biological enrichment.  
Values of 8x are observed here, while the true enrichment (binary) is only ~1.1x.

In [73]:
# Build normalized transactions — exactly as the current algorithm does

transactions_norm = []

for center_i, idxs in enumerate(neighbors_idx):
    neighbor_idxs = [n for n in idxs if n != center_i]
    if len(neighbor_idxs) == 0:
        continue
    if is_dominated(cell_types[list(idxs)]):
        continue

    trans = {}
    for n_idx in neighbor_idxs:
        d = np.linalg.norm(coords[n_idx] - coords[center_i])
        w = np.exp(-0.5 * (d / BANDWIDTH) ** 2)  # Gaussian weight
        ct = f"{cell_types[n_idx]}_NEIGHBOR"
        trans[ct] = trans.get(ct, 0.0) + w

    trans[f"{cell_types[center_i]}_CENTER"] = 1.0  # center weight = 1.0 before normalization

    # Normalize: divide EVERYTHING (including center) by the total
    total = sum(trans.values())
    print("Total sum of weights before normalization:", total)
    trans = {k: v / total for k, v in trans.items()}

    transactions_norm.append(trans)

print(f"Total transactions: {len(transactions_norm)}")

Total sum of weights before normalization: 12.36941957389923
Total sum of weights before normalization: 12.007521045795178
Total sum of weights before normalization: 12.56547667390615
Total sum of weights before normalization: 13.892472664432308
Total sum of weights before normalization: 14.094011048416382
Total sum of weights before normalization: 13.013114504496784
Total sum of weights before normalization: 11.607665268511827
Total sum of weights before normalization: 14.107754773813145
Total sum of weights before normalization: 13.88216873151849
Total sum of weights before normalization: 11.635027713406275
Total sum of weights before normalization: 6.704308001526488
Total sum of weights before normalization: 11.16178495474648
Total sum of weights before normalization: 13.520877949333894
Total sum of weights before normalization: 15.257386568461461
Total sum of weights before normalization: 11.142457692394295
Total sum of weights before normalization: 15.350157880025758
Total sum of 

In [74]:
# Show the dilution effect in a dense neighborhood

# Find a dense Neutrophil neighborhood (many Neutrophil neighbors)
raw_examples = []
for center_i, idxs in enumerate(neighbors_idx):
    if cell_types[center_i] != "Neutrophil" or len(idxs) < 8:
        continue
    neighbor_idxs = [n for n in idxs if n != center_i]
    trans_raw = {}
    for n_idx in neighbor_idxs:
        d = np.linalg.norm(coords[n_idx] - coords[center_i])
        w = np.exp(-0.5 * (d / BANDWIDTH) ** 2)
        ct = f"{cell_types[n_idx]}_NEIGHBOR"
        trans_raw[ct] = trans_raw.get(ct, 0.0) + w
    trans_raw["Neutrophil_CENTER"] = 1.0
    total = sum(trans_raw.values())
    raw_examples.append((center_i, total, trans_raw))

center_i, total, trans_raw = max(raw_examples, key=lambda x: x[1])

print(f"Dense Neutrophil neighborhood (cell {center_i}, {len(trans_raw)-1} neighbor types):")
print(f"  Sum of all raw Gaussian weights (total) = {total:.2f}")
print()
print("  {:40}  {:>10}  {:>12}".format("Item", "Raw weight", "After /total"))
print("  " + "-"*66)
for k in sorted(trans_raw, key=lambda k: trans_raw[k], reverse=True):
    raw = trans_raw[k]
    print(f"  {k:<40}  {raw:>10.3f}  {raw/total:>12.4f}")
print()
print(f"-> Neutrophil_CENTER: 1.0 (raw) -> {1.0/total:.4f} (after normalization)")
print(f"-> In a dense area the center cell collapses, dragging support(CENTER) far below the true frequency")

Dense Neutrophil neighborhood (cell 1496, 1 neighbor types):
  Sum of all raw Gaussian weights (total) = 26.73

  Item                                      Raw weight  After /total
  ------------------------------------------------------------------
  Muscle_NEIGHBOR                               25.729        0.9626
  Neutrophil_CENTER                              1.000        0.0374

-> Neutrophil_CENTER: 1.0 (raw) -> 0.0374 (after normalization)
-> In a dense area the center cell collapses, dragging support(CENTER) far below the true frequency


In [75]:
# Show the bug: normalized support(FibroticEpithelial_CENTER) vs true frequency

N = len(transactions_norm)
true_fibro_freq  = sum(1 for t in transactions_norm if "FibroticEpithelial_CENTER" in t) / N
norm_fibro_supp  = sum(t.get("FibroticEpithelial_CENTER", 0) for t in transactions_norm) / N

print(f"True FibroticEpithelial frequency:              {true_fibro_freq:.4f}  ({true_fibro_freq*N:.0f}/{N} cells)")
print(f"Normalized support(FibroticEpithelial_CENTER):  {norm_fibro_supp:.4f}")
print(f"Sum weights of FibroticEpithelial_CENTER across all transactions: {sum(t.get("FibroticEpithelial_CENTER", 0) for t in transactions_norm):.2f}")
print()
print(f"-> Normalized support is {true_fibro_freq/norm_fibro_supp:.1f}x lower than the true frequency")
print(f"-> Because Fibrotic cells are surrounded by dense Neutrophil clusters")
print(f"-> MIN_SUPPORT threshold (e.g. 0.01) will filter out Fibrotic rules even though")
print(f"   there are {true_fibro_freq*N:.0f} Fibrotic cells ({true_fibro_freq*100:.1f}% of the FOV)")

True FibroticEpithelial frequency:              0.0916  (172/1877 cells)
Normalized support(FibroticEpithelial_CENTER):  0.0075
Sum weights of FibroticEpithelial_CENTER across all transactions: 14.06

-> Normalized support is 12.2x lower than the true frequency
-> Because Fibrotic cells are surrounded by dense Neutrophil clusters
-> MIN_SUPPORT threshold (e.g. 0.01) will filter out Fibrotic rules even though
   there are 172 Fibrotic cells (9.2% of the FOV)


In [76]:
# Rule metrics with normalization
m1_n = compute_metrics(transactions_norm, "Neutrophil_CENTER",         "FibroticEpithelial_NEIGHBOR")
m2_n = compute_metrics(transactions_norm, "FibroticEpithelial_CENTER", "Neutrophil_NEIGHBOR")
m3_n = compute_metrics(transactions_norm, "Macrophage_CENTER",         "Macrophage_NEIGHBOR")

print("=" * 60)
print_metrics(m1_n, "Neutrophil_CENTER", "FibroticEpithelial_NEIGHBOR")
print()
print("=" * 60)
print_metrics(m2_n, "FibroticEpithelial_CENTER", "Neutrophil_NEIGHBOR")
print()
print("=" * 60)
print_metrics(m3_n, "Macrophage_CENTER", "Macrophage_NEIGHBOR")

Rule:  Neutrophil_CENTER
    ->  FibroticEpithelial_NEIGHBOR
  support(A)   = 0.0161  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 0.0821  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.0112  (avg min(w_A, w_B) per transaction; binary: fraction where both co-occur)
  confidence   = 0.700   (support(A,B) / support(A); binary: P(B present | A is CENTER) = 70.0%)
  lift         = 8.53   (confidence / support(B); >1 means co-occurrence stronger than chance)
  conviction   = 3.06  (how often A occurs without B being incorrectly predicted absent; >1 means rule is informative)

Rule:  FibroticEpithelial_CENTER
    ->  Neutrophil_NEIGHBOR
  support(A)   = 0.0075  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 0.2444  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.0063  (avg min(w_A, w_B) per transac

In [77]:
# Show the inflation: compare binary lift vs normalized lift
print("LIFT INFLATION: Binary (ground truth) vs Normalized")
print("-" * 50)
print(f"Neutrophil -> Fibrotic")
print(f"  Binary:      lift = {m1['lift']:.2f}  (true spatial enrichment)")
print(f"  Normalized:  lift = {m1_n['lift']:.2f}  (inflated by normalization)")
print(f"  Inflation:   {m1_n['lift']/m1['lift']:.1f}x")
print()
print(f"Macrophage -> Macrophage")
print(f"  Binary:      lift = {m3['lift']:.2f}")
print(f"  Normalized:  lift = {m3_n['lift']:.2f}")
print(f"  Inflation:   {m3_n['lift']/m3['lift']:.1f}x")
print()
print("-> Normalization inflates lift across ALL rules systematically")
print("-> The missing Fibrotic rules are not just due to low lift — their support is below the threshold")

LIFT INFLATION: Binary (ground truth) vs Normalized
--------------------------------------------------
Neutrophil -> Fibrotic
  Binary:      lift = 1.11  (true spatial enrichment)
  Normalized:  lift = 8.53  (inflated by normalization)
  Inflation:   7.7x

Macrophage -> Macrophage
  Binary:      lift = 1.08
  Normalized:  lift = 7.16
  Inflation:   6.6x

-> Normalization inflates lift across ALL rules systematically
-> The missing Fibrotic rules are not just due to low lift — their support is below the threshold


---
## Section 3 — Capping Approach: Cap Gaussian Weights at 1.0

### What is it?
Same Gaussian decay as Section 2, but **no global normalization**.  
Instead, each cell type's weight is independently capped at 1.0:  
`weight = min(1.0, sum of Gaussian weights for that cell type)`

- **CENTER** is always 1.0 — a cell is always "fully present" as its own center, regardless of neighborhood density
- **NEIGHBOR** in [0, 1] — measures how strongly a cell type is present nearby based on closeness.  
  Multiple close cells of the same type saturate at 1.0 ("receptor saturation")

---

### Metric Definitions (for this biological case)

**support(FibroticEpithelial_CENTER) = true frequency**  
= count(Fibrotic-centered transactions) / N  
Always equals the true cell type frequency — no longer diluted by dense neighborhoods.

**support(FibroticEpithelial_NEIGHBOR)**  
= average capped Gaussian signal of Fibrotic cells across ALL neighborhoods  
Captures both "is a Fibrotic cell nearby?" and "how close is it?" — independently of other cell types.

**confidence(Neutrophil -> Fibrotic)**  
= average capped Fibrotic signal in Neutrophil-centered neighborhoods  
*"How strongly present are Fibrotic cells in an average Neutrophil neighborhood?"*

**lift(A -> B)**  
= (B's average Gaussian signal in A-centered neighborhoods) / (B's average signal in all neighborhoods)  
*"Is B specifically denser or closer in A's neighborhood compared to a random neighborhood?"*  
This is the **most accurate** measure — preserves distance sensitivity without normalization artifacts.

**conviction(A -> B)**  
= (1 - support(B)) / (1 - confidence)  
With capping, support(B) is a meaningful average signal in [0,1], so conviction is well-calibrated.

In [78]:
# Build capped transactions

transactions_cap = []

for center_i, idxs in enumerate(neighbors_idx):
    neighbor_idxs = [n for n in idxs if n != center_i]
    if len(neighbor_idxs) == 0:
        continue
    if is_dominated(cell_types[list(idxs)]):
        continue

    trans = {}
    for n_idx in neighbor_idxs:
        d = np.linalg.norm(coords[n_idx] - coords[center_i])
        w = np.exp(-0.5 * (d / BANDWIDTH) ** 2)  # Gaussian weight
        ct = f"{cell_types[n_idx]}_NEIGHBOR"
        trans[ct] = trans.get(ct, 0.0) + w

    # Cap each cell type independently at 1.0 — no normalization
    trans = {k: min(1.0, v) for k, v in trans.items()}
    # Normalized: trans = {k: v / total for k, v in trans.items()}
    trans[f"{cell_types[center_i]}_CENTER"] = 1.0  # always 1.0

    transactions_cap.append(trans)

print(f"Total transactions: {len(transactions_cap)}")

Total transactions: 1877


In [79]:
# Compare the same Neutrophil-centered transaction across all three approaches

# Find a Neutrophil cell that has some FibroticEpithelial neighbors
target_center = None
for center_i, idxs in enumerate(neighbors_idx):
    if cell_types[center_i] != "Neutrophil":
        continue
    types_in_hood = [cell_types[n] for n in idxs if n != center_i]
    if "FibroticEpithelial" in types_in_hood and len(types_in_hood) >= 5:
        target_center = center_i
        break

print(f"Neutrophil cell {target_center} with FibroticEpithelial neighbors")
print(f"Neighbor cell types: {pd.Series([cell_types[n] for n in neighbors_idx[target_center] if n != target_center]).value_counts().to_dict()}")
print()

# Get this cell's transaction from each list
def get_nth_matching(transactions, center_type, neighbor_type):
    return next((t for t in transactions
                 if f"{center_type}_CENTER" in t and f"{neighbor_type}_NEIGHBOR" in t), None)

tb = get_nth_matching(transactions_binary, "Neutrophil", "FibroticEpithelial")
tn = get_nth_matching(transactions_norm,   "Neutrophil", "FibroticEpithelial")
tc = get_nth_matching(transactions_cap,    "Neutrophil", "FibroticEpithelial")

all_keys = sorted(set(list(tb.keys()) + list(tn.keys()) + list(tc.keys())))
print("  {:45} {:>8} {:>8} {:>8}".format("Item", "Binary", "Norm", "Capped"))
print("  " + "-" * 73)
for k in all_keys:
    print(f"  {k:<45} {tb.get(k,0):>8.4f} {tn.get(k,0):>8.4f} {tc.get(k,0):>8.4f}")

Neutrophil cell 1495 with FibroticEpithelial neighbors
Neighbor cell types: {'Macrophage': 12, 'Muscle': 6, 'Fibroblast': 6, 'Neuron': 2, 'Neutrophil_CD15': 1, 'FibroticEpithelial': 1, 'CD8T': 1}

  Item                                            Binary     Norm   Capped
  -------------------------------------------------------------------------
  CD8T_NEIGHBOR                                   1.0000   0.0277   0.4378
  Fibroblast_NEIGHBOR                             1.0000   0.2035   1.0000
  FibroticEpithelial_NEIGHBOR                     1.0000   0.0297   0.4690
  Macrophage_NEIGHBOR                             1.0000   0.4037   1.0000
  Muscle_NEIGHBOR                                 1.0000   0.1342   1.0000
  Neuron_NEIGHBOR                                 1.0000   0.0806   1.0000
  Neutrophil_CD15_NEIGHBOR                        1.0000   0.0572   0.9025
  Neutrophil_CENTER                               1.0000   0.0634   1.0000


In [80]:
# Show that capping restores the correct support for FibroticEpithelial_CENTER
N = len(transactions_cap)

true_freq  = sum(1 for t in transactions_cap if "FibroticEpithelial_CENTER" in t) / N
cap_supp   = sum(t.get("FibroticEpithelial_CENTER", 0) for t in transactions_cap) / N

print(f"True FibroticEpithelial frequency:              {true_freq:.4f}  ({true_freq*N:.0f}/{N} cells)")
print(f"Capped support(FibroticEpithelial_CENTER):      {cap_supp:.4f}")
print()
print(f"-> They are equal! CENTER = 1.0 always, so support(CENTER) = cell type frequency.")
print(f"-> No distortion from neighborhood density.")

True FibroticEpithelial frequency:              0.0916  (172/1877 cells)
Capped support(FibroticEpithelial_CENTER):      0.0916

-> They are equal! CENTER = 1.0 always, so support(CENTER) = cell type frequency.
-> No distortion from neighborhood density.


In [81]:
# Rule metrics with capping
m1_c = compute_metrics(transactions_cap, "Neutrophil_CENTER",         "FibroticEpithelial_NEIGHBOR")
m2_c = compute_metrics(transactions_cap, "FibroticEpithelial_CENTER", "Neutrophil_NEIGHBOR")
m3_c = compute_metrics(transactions_cap, "Macrophage_CENTER",         "Macrophage_NEIGHBOR")

print("=" * 60)
print_metrics(m1_c, "Neutrophil_CENTER", "FibroticEpithelial_NEIGHBOR")
print()
print("=" * 60)
print_metrics(m2_c, "FibroticEpithelial_CENTER", "Neutrophil_NEIGHBOR")
print()
print("=" * 60)
print_metrics(m3_c, "Macrophage_CENTER", "Macrophage_NEIGHBOR")

Rule:  Neutrophil_CENTER
    ->  FibroticEpithelial_NEIGHBOR
  support(A)   = 0.2547  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 0.6034  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.1690  (avg min(w_A, w_B) per transaction; binary: fraction where both co-occur)
  confidence   = 0.663   (support(A,B) / support(A); binary: P(B present | A is CENTER) = 66.3%)
  lift         = 1.10   (confidence / support(B); >1 means co-occurrence stronger than chance)
  conviction   = 1.18  (how often A occurs without B being incorrectly predicted absent; >1 means rule is informative)

Rule:  FibroticEpithelial_CENTER
    ->  Neutrophil_NEIGHBOR
  support(A)   = 0.0916  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 0.6978  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.0811  (avg min(w_A, w_B) per transac

---
## Section 4 — Raw Gaussian Weights (No Normalization, No Capping)

### What is it?
Each cell's transaction uses the **raw sum of Gaussian weights** for each neighbor type:
- `CENTER` = 1.0 (always)
- `NEIGHBOR_type` = Σ exp(−0.5 × (d/σ)²) for all cells of that type within radius

No normalization, no cap. If 10 Neutrophils are clustered nearby, `Neutrophil_NEIGHBOR` weight can be 5.0 or more.

### Metric interpretation

**support(A_CENTER)**  
= fraction of all transactions where A is the center cell (always = cell type frequency, same as binary)

**support(B_NEIGHBOR)**  
= average raw Gaussian signal of B across all transactions  
⚠️ Can be **> 1** for abundant neighbor types — it is no longer a probability!

**confidence(A → B)**  
= average raw signal of B in A-centered transactions / average raw signal of A  
Interpretable as a relative enrichment, but unbounded.

**lift(A → B)**  
= confidence / support(B)  
Still tells you whether B is enriched in A's neighborhood vs. a random neighborhood.  
The scale cancels out, so lift remains comparable across methods.

**conviction(A → B)**  
= (1 − support(B)) / (1 − confidence)  
⚠️ Breaks when support(B) > 1 or confidence > 1 — may be negative or undefined.

In [82]:
# Build raw-weighted transactions — no normalization, no capping

transactions_raw = []

for center_i, idxs in enumerate(neighbors_idx):
    neighbor_idxs = [n for n in idxs if n != center_i]
    if len(neighbor_idxs) == 0:
        continue
    if is_dominated(cell_types[list(idxs)]):
        continue

    trans = {}
    for n_idx in neighbor_idxs:
        d = np.linalg.norm(coords[n_idx] - coords[center_i])
        w = np.exp(-0.5 * (d / BANDWIDTH) ** 2)  # raw Gaussian weight
        ct = f"{cell_types[n_idx]}_NEIGHBOR"
        trans[ct] = trans.get(ct, 0.0) + w  # accumulate, do NOT normalize or cap

    trans[f"{cell_types[center_i]}_CENTER"] = 1.0  # center always 1.0

    transactions_raw.append(trans)

print(f"Total transactions: {len(transactions_raw)}")

Total transactions: 1877


In [83]:
# Show example: same Neutrophil cell as before, raw weights can exceed 1
tr = get_nth_matching(transactions_raw, "Neutrophil", "FibroticEpithelial")

all_keys = sorted(set(list(tb.keys()) + list(tr.keys())))
print("  {:45} {:>8} {:>10}".format("Item", "Binary", "Raw"))
print("  " + "-" * 67)
for k in all_keys:
    rv = tr.get(k, 0)
    flag = "  <- > 1!" if rv > 1.0 else ""
    print(f"  {k:<45} {tb.get(k,0):>8.4f} {rv:>10.4f}{flag}")
print()
print("Note: NEIGHBOR weights > 1 mean multiple cells of that type are in the neighborhood")

  Item                                            Binary        Raw
  -------------------------------------------------------------------
  CD8T_NEIGHBOR                                   1.0000     0.4378
  Fibroblast_NEIGHBOR                             1.0000     3.2117  <- > 1!
  FibroticEpithelial_NEIGHBOR                     1.0000     0.4690
  Macrophage_NEIGHBOR                             1.0000     6.3718  <- > 1!
  Muscle_NEIGHBOR                                 1.0000     2.1183  <- > 1!
  Neuron_NEIGHBOR                                 1.0000     1.2727  <- > 1!
  Neutrophil_CD15_NEIGHBOR                        1.0000     0.9025
  Neutrophil_CENTER                               1.0000     1.0000

Note: NEIGHBOR weights > 1 mean multiple cells of that type are in the neighborhood


In [84]:
# Show support values — CENTER support = true frequency, NEIGHBOR support can be > 1
N = len(transactions_raw)

fibro_center_supp = sum(t.get("FibroticEpithelial_CENTER", 0) for t in transactions_raw) / N
neut_center_supp  = sum(t.get("Neutrophil_CENTER", 0)        for t in transactions_raw) / N
neut_neighbor_supp = sum(t.get("Neutrophil_NEIGHBOR", 0)     for t in transactions_raw) / N
true_fibro_freq    = sum(1 for t in transactions_raw if "FibroticEpithelial_CENTER" in t) / N

print(f"support(FibroticEpithelial_CENTER)  = {fibro_center_supp:.4f}  (true freq = {true_fibro_freq:.4f} — they match, CENTER=1 always)")
print(f"support(Neutrophil_CENTER)          = {neut_center_supp:.4f}")
print(f"support(Neutrophil_NEIGHBOR)        = {neut_neighbor_supp:.4f}  <- can be > 1 for abundant cell types")

support(FibroticEpithelial_CENTER)  = 0.0916  (true freq = 0.0916 — they match, CENTER=1 always)
support(Neutrophil_CENTER)          = 0.2547
support(Neutrophil_NEIGHBOR)        = 4.1339  <- can be > 1 for abundant cell types


In [85]:
# Rule metrics with raw weights
m1_r = compute_metrics(transactions_raw, "Neutrophil_CENTER",         "FibroticEpithelial_NEIGHBOR")
m2_r = compute_metrics(transactions_raw, "FibroticEpithelial_CENTER", "Neutrophil_NEIGHBOR")
m3_r = compute_metrics(transactions_raw, "Macrophage_CENTER",         "Macrophage_NEIGHBOR")

print("=" * 60)
print_metrics(m1_r, "Neutrophil_CENTER", "FibroticEpithelial_NEIGHBOR")
print()
print("=" * 60)
print_metrics(m2_r, "FibroticEpithelial_CENTER", "Neutrophil_NEIGHBOR")
print()
print("=" * 60)
print_metrics(m3_r, "Macrophage_CENTER", "Macrophage_NEIGHBOR")
print()
print("Note: conviction may show 'inf' or negative values when support(B) > 1 — this metric breaks for unbounded weights")

Rule:  Neutrophil_CENTER
    ->  FibroticEpithelial_NEIGHBOR
  support(A)   = 0.2547  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 1.1155  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.1690  (avg min(w_A, w_B) per transaction; binary: fraction where both co-occur)
  confidence   = 0.663   (support(A,B) / support(A); binary: P(B present | A is CENTER) = 66.3%)
  lift         = 0.59   (confidence / support(B); >1 means co-occurrence stronger than chance)
  conviction   = -0.34  (how often A occurs without B being incorrectly predicted absent; >1 means rule is informative)

Rule:  FibroticEpithelial_CENTER
    ->  Neutrophil_NEIGHBOR
  support(A)   = 0.0916  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 4.1339  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.0811  (avg min(w_A, w_B) per transa

---
## Section 5 — Log Per-Cell-Type Rescaling (Receptor Saturation Model)

### What is it?
A two-pass approach that normalizes each cell type by its own density baseline **within this FOV**:

1. **Pass 1 (per-FOV):** compute the 90th percentile of raw Gaussian sums for each cell type independently — call it `P90_C`.
2. **Pass 2:** apply `W_final = min(1.0, log(1 + W_raw) / log(1 + P90_C))`

The result is strictly in (0, 1]. The log function compresses high signals (dense clusters) while keeping rare/sparse signals proportional.

### Metric interpretation

**support(A_CENTER)** = fraction of transactions where A is the center cell (always = cell type frequency, CENTER=1.0 always)

**support(B_NEIGHBOR)** = average log-rescaled signal of B per transaction; in [0,1] because of the min(1.0,...) cap.  
Meaning: *"how often does B appear at above-typical density in an average neighborhood?"*

**confidence(A → B)** = avg min(w_A, w_B) in A-centered transactions / avg w_A.  
Meaning: *"in A-centered neighborhoods, how much of B's typical signal is present?"*

**lift(A → B)** = confidence / support(B); still tells you whether B is enriched in A's neighborhood vs random.

**conviction** = (1 − support(B)) / (1 − confidence); valid here since all weights ∈ (0, 1].

### ⚠️ Known issues with this approach (shown below)
- Cell types with **P90 < 1.0** are effectively binary: a single nearby cell already saturates the log-rescaled weight to 1.0
- Creates **asymmetry across cell types**: abundant types (Neutrophil, P90=13) get low weight per cell; sparse types (CD4T, P90=0.87) get weight ≈ 1.0 per cell — purely because of FOV-level abundance differences, not biology
- This can **inflate lift for rules involving rare cell types** relative to abundant ones, independent of actual spatial enrichment

In [86]:
# Pass 1: compute P90 per cell type from this FOV's raw transactions
# (P90 is computed per-FOV because rule mining runs per-FOV)

from collections import defaultdict

raw_by_type = defaultdict(list)
for t in transactions_raw:
    for k, v in t.items():
        if k.endswith('_NEIGHBOR'):
            ct = k[:-len('_NEIGHBOR')]
            raw_by_type[ct].append(v)

p90_per_type = {ct: np.percentile(vals, 90) for ct, vals in raw_by_type.items()}

print('P90 per cell type (this FOV):')
print('{:<30} {:>8}  {}'.format('Cell type', 'P90', 'Note'))
print('-' * 60)
for ct, p in sorted(p90_per_type.items(), key=lambda x: x[1]):
    note = '<-- P90 < 1: effectively binary (single close cell = max weight)' if p < 1.0 else ''
    print('{:<30} {:>8.3f}  {}'.format(ct, p, note))

P90 per cell type (this FOV):
Cell type                           P90  Note
------------------------------------------------------------
Plasma_CD38                       0.782  <-- P90 < 1: effectively binary (single close cell = max weight)
CD3T                              0.816  <-- P90 < 1: effectively binary (single close cell = max weight)
Plasma                            0.819  <-- P90 < 1: effectively binary (single close cell = max weight)
Monocyte                          0.823  <-- P90 < 1: effectively binary (single close cell = max weight)
ImmuneOther                       0.828  <-- P90 < 1: effectively binary (single close cell = max weight)
Unidentified                      0.849  <-- P90 < 1: effectively binary (single close cell = max weight)
Epithelial                        0.877  <-- P90 < 1: effectively binary (single close cell = max weight)
Mesenchymal_VIM                   0.887  <-- P90 < 1: effectively binary (single close cell = max weight)
CD4T           

In [87]:
# Pass 2: build log-rescaled transactions

transactions_log = []

for center_i, idxs in enumerate(neighbors_idx):
    neighbor_idxs = [n for n in idxs if n != center_i]
    if len(neighbor_idxs) == 0:
        continue
    if is_dominated(cell_types[list(idxs)]):
        continue

    trans = {}
    for n_idx in neighbor_idxs:
        d = np.linalg.norm(coords[n_idx] - coords[center_i])
        w = np.exp(-0.5 * (d / BANDWIDTH) ** 2)  # raw Gaussian
        ct = f"{cell_types[n_idx]}_NEIGHBOR"
        trans[ct] = trans.get(ct, 0.0) + w

    # Log rescaling per cell type using this FOV's P90
    trans_log = {}
    for k, w_raw in trans.items():
        ct = k[:-len('_NEIGHBOR')]
        p90_ct = p90_per_type.get(ct, 1.0)
        trans_log[k] = min(1.0, np.log1p(w_raw) / np.log1p(p90_ct))

    trans_log[f"{cell_types[center_i]}_CENTER"] = 1.0
    transactions_log.append(trans_log)

print(f'Total transactions: {len(transactions_log)}')

Total transactions: 1877


In [88]:
# PROBLEM 1: Weight asymmetry across cell types
# The same biological situation — 1 cell at 5 microns — gets very different weights
# depending solely on how abundant that cell type is in this FOV.

d_close = 5.0   # microns — a close neighbor
d_far   = 20.0  # microns — a distant neighbor
w_close = np.exp(-0.5 * (d_close / BANDWIDTH) ** 2)  # ≈ 0.94
w_far   = np.exp(-0.5 * (d_far   / BANDWIDTH) ** 2)  # ≈ 0.25

print('Single-cell log weight vs Binary weight:')
print('(same number of cells, different abundance in FOV)')
print()
print('{:<22} {:>8} {:>10} {:>10} {:>10}'.format('Cell type', 'P90', 'Log@5um', 'Log@20um', 'Binary'))
print('-' * 65)
for ct in ['Muscle', 'Neutrophil', 'Macrophage', 'FibroticEpithelial', 'CD8T', 'CD4T', 'Epithelial']:
    p = p90_per_type.get(ct, 1.0)
    wl_close = min(1.0, np.log1p(w_close) / np.log1p(p))
    wl_far   = min(1.0, np.log1p(w_far)   / np.log1p(p))
    print('{:<22} {:>8.3f} {:>10.3f} {:>10.3f} {:>10}'.format(ct, p, wl_close, wl_far, '1 / absent'))
print()
print('-> 1 CD4T at 5um = weight 1.0; 1 Neutrophil at 5um = weight ~0.24')
print('-> CD4T is weighted ~4x more than Neutrophil for equal proximity')
print('   This is a pure abundance artifact, not a biological signal')

Single-cell log weight vs Binary weight:
(same number of cells, different abundance in FOV)

Cell type                   P90    Log@5um   Log@20um     Binary
-----------------------------------------------------------------
Muscle                   24.573      0.205      0.106 1 / absent
Neutrophil               13.273      0.250      0.130 1 / absent
Macrophage                4.658      0.384      0.199 1 / absent
FibroticEpithelial        3.018      0.479      0.248 1 / absent
CD8T                      2.806      0.498      0.258 1 / absent
CD4T                      0.889      1.000      0.541 1 / absent
Epithelial                0.877      1.000      0.547 1 / absent

-> 1 CD4T at 5um = weight 1.0; 1 Neutrophil at 5um = weight ~0.24
-> CD4T is weighted ~4x more than Neutrophil for equal proximity
   This is a pure abundance artifact, not a biological signal


In [89]:
# PROBLEM 2: Rules involving rare cell types (low P90) inflate lift
# Compare lift for rules with abundant vs rare neighbor types across approaches

# Additional rules involving sparse cell types
sparse_types = [ct for ct, p in sorted(p90_per_type.items(), key=lambda x: x[1]) if p < 1.2]
print('Cell types with P90 < 1.2 (borderline binary after log rescaling):')
for ct in sparse_types:
    print(f'  {ct:<28}  P90 = {p90_per_type[ct]:.3f}')
print()

# Show lift for a rule involving a sparse type: Macrophage -> CD4T_NEIGHBOR
# Compare: should be similar to binary if CD4T is sparse (P90 < 1)
mb_cd4 = compute_metrics(transactions_binary, 'Macrophage_CENTER', 'CD4T_NEIGHBOR')
ml_cd4 = compute_metrics(transactions_log,    'Macrophage_CENTER', 'CD4T_NEIGHBOR')
mc_cd4 = compute_metrics(transactions_cap,    'Macrophage_CENTER', 'CD4T_NEIGHBOR')

print('Rule: Macrophage_CENTER -> CD4T_NEIGHBOR  (CD4T has P90={:.3f})'.format(p90_per_type.get('CD4T', 0)))
print('  lift (binary) = {:.3f}'.format(mb_cd4['lift']))
print('  lift (log)    = {:.3f}  <- should be close to binary since CD4T P90 < 1'.format(ml_cd4['lift']))
print('  lift (capped) = {:.3f}'.format(mc_cd4['lift']))
print()

# Now compare vs a rule involving abundant Neutrophil
mb_neut = compute_metrics(transactions_binary, 'Macrophage_CENTER', 'Neutrophil_NEIGHBOR')
ml_neut = compute_metrics(transactions_log,    'Macrophage_CENTER', 'Neutrophil_NEIGHBOR')
mc_neut = compute_metrics(transactions_cap,    'Macrophage_CENTER', 'Neutrophil_NEIGHBOR')

print('Rule: Macrophage_CENTER -> Neutrophil_NEIGHBOR  (Neutrophil has P90={:.3f})'.format(p90_per_type.get('Neutrophil', 0)))
print('  lift (binary) = {:.3f}'.format(mb_neut['lift']))
print('  lift (log)    = {:.3f}  <- log suppresses Neutrophil weight -> inflates lift relative to binary'.format(ml_neut['lift']))
print('  lift (capped) = {:.3f}'.format(mc_neut['lift']))
print()
print('Conclusion: log-rescaling makes CD4T-involving rules look comparably strong to Neutrophil-involving')
print('rules even when the actual binary enrichment is the same. The difference is a FOV-abundance artifact.')

Cell types with P90 < 1.2 (borderline binary after log rescaling):
  Plasma_CD38                   P90 = 0.782
  CD3T                          P90 = 0.816
  Plasma                        P90 = 0.819
  Monocyte                      P90 = 0.823
  ImmuneOther                   P90 = 0.828
  Unidentified                  P90 = 0.849
  Epithelial                    P90 = 0.877
  Mesenchymal_VIM               P90 = 0.887
  CD4T                          P90 = 0.889
  ImmuneOther_CD45RA            P90 = 0.919
  Mast                          P90 = 0.935
  Treg                          P90 = 0.977

Rule: Macrophage_CENTER -> CD4T_NEIGHBOR  (CD4T has P90=0.889)
  lift (binary) = 0.969
  lift (log)    = 1.006  <- should be close to binary since CD4T P90 < 1
  lift (capped) = 1.025

Rule: Macrophage_CENTER -> Neutrophil_NEIGHBOR  (Neutrophil has P90=13.273)
  lift (binary) = 1.109
  lift (log)    = 0.822  <- log suppresses Neutrophil weight -> inflates lift relative to binary
  lift (capped) = 1.05

In [90]:
# Rule metrics with log rescaling — main rules
m1_l = compute_metrics(transactions_log, 'Neutrophil_CENTER',         'FibroticEpithelial_NEIGHBOR')
m2_l = compute_metrics(transactions_log, 'FibroticEpithelial_CENTER', 'Neutrophil_NEIGHBOR')
m3_l = compute_metrics(transactions_log, 'Macrophage_CENTER',         'Macrophage_NEIGHBOR')

print('=' * 60)
print_metrics(m1_l, 'Neutrophil_CENTER', 'FibroticEpithelial_NEIGHBOR')
print()
print('=' * 60)
print_metrics(m2_l, 'FibroticEpithelial_CENTER', 'Neutrophil_NEIGHBOR')
print()
print('=' * 60)
print_metrics(m3_l, 'Macrophage_CENTER', 'Macrophage_NEIGHBOR')

Rule:  Neutrophil_CENTER
    ->  FibroticEpithelial_NEIGHBOR
  support(A)   = 0.2547  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 0.4365  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.1212  (avg min(w_A, w_B) per transaction; binary: fraction where both co-occur)
  confidence   = 0.476   (support(A,B) / support(A); binary: P(B present | A is CENTER) = 47.6%)
  lift         = 1.09   (confidence / support(B); >1 means co-occurrence stronger than chance)
  conviction   = 1.08  (how often A occurs without B being incorrectly predicted absent; >1 means rule is informative)

Rule:  FibroticEpithelial_CENTER
    ->  Neutrophil_NEIGHBOR
  support(A)   = 0.0916  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 0.4433  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.0444  (avg min(w_A, w_B) per transac

---
## Section 6 — P90 Cap Per Cell Type (No Normalization)

### What is it?
Use the raw Gaussian sum but clip each cell type at its own FOV P90:

```
W_final = min(P90_C, W_raw)
```

P90_C comes from the same FOV (already computed in Section 5 Pass 1). No division — the scale is untouched.

### Why?
- Removes the top-10% extreme outliers that can dominate support per cell type
- Keeps the raw Gaussian scale otherwise — distance information fully preserved
- Cell types are still on their own natural density scales (Neutrophil ~0–13, CD4T ~0–0.87)

### ⚠️ Caveat
Since P90_C > 1 for abundant types, **support(NEIGHBOR) can still exceed 1** for Neutrophil, Muscle, etc.  
This means conviction and confidence lose their probability interpretation for those cell types — same limitation as Section 4 (Raw).  
**Lift is still valid**: it is a ratio of supports, so the > 1 issue cancels out.

In [91]:
# Build P90-capped transactions (reuses p90_per_type from sect5-p90)
transactions_p90cap = []

for center_i, idxs in enumerate(neighbors_idx):
    neighbor_idxs = [n for n in idxs if n != center_i]
    if len(neighbor_idxs) == 0:
        continue
    if is_dominated(cell_types[list(idxs)]):
        continue

    trans = {}
    for n_idx in neighbor_idxs:
        d = np.linalg.norm(coords[n_idx] - coords[center_i])
        w = np.exp(-0.5 * (d / BANDWIDTH) ** 2)
        ct = f"{cell_types[n_idx]}_NEIGHBOR"
        trans[ct] = trans.get(ct, 0.0) + w

    # Clip each cell type at its own FOV P90
    trans_p90 = {}
    for k, w_raw in trans.items():
        ct = k[:-len('_NEIGHBOR')]
        p90_ct = p90_per_type.get(ct, w_raw)  # fallback: no clip if unseen
        trans_p90[k] = min(p90_ct, w_raw)

    trans_p90[f"{cell_types[center_i]}_CENTER"] = 1.0
    transactions_p90cap.append(trans_p90)

print(f'Total transactions: {len(transactions_p90cap)}')

# Show what the weights look like vs raw for the two key cell types
neut_raw  = [t.get('Neutrophil_NEIGHBOR', 0) for t in transactions_raw]
neut_cap  = [t.get('Neutrophil_NEIGHBOR', 0) for t in transactions_p90cap]
fibro_raw = [t.get('FibroticEpithelial_NEIGHBOR', 0) for t in transactions_raw]
fibro_cap = [t.get('FibroticEpithelial_NEIGHBOR', 0) for t in transactions_p90cap]

print(f"\nNeutrophil NEIGHBOR  — raw mean: {np.mean(neut_raw):.3f}, p90cap mean: {np.mean(neut_cap):.3f}  (P90={p90_per_type.get('Neutrophil',0):.2f})")
print(f"FibroticEpi NEIGHBOR — raw mean: {np.mean(fibro_raw):.3f}, p90cap mean: {np.mean(fibro_cap):.3f}  (P90={p90_per_type.get('FibroticEpithelial',0):.2f})")
print(f"\n-> For FibroticEpithelial (P90={p90_per_type.get('FibroticEpithelial',0):.2f} < 1): clipping rarely fires, values nearly identical to raw")
print(f"-> For Neutrophil (P90={p90_per_type.get('Neutrophil',0):.2f}): clips the densest transactions, flattening extremes")

Total transactions: 1877

Neutrophil NEIGHBOR  — raw mean: 4.134, p90cap mean: 3.894  (P90=13.27)
FibroticEpi NEIGHBOR — raw mean: 1.115, p90cap mean: 1.069  (P90=3.02)

-> For FibroticEpithelial (P90=3.02 < 1): clipping rarely fires, values nearly identical to raw
-> For Neutrophil (P90=13.27): clips the densest transactions, flattening extremes


In [92]:
# Rule metrics with P90-cap per cell type
m1_p = compute_metrics(transactions_p90cap, 'Neutrophil_CENTER',         'FibroticEpithelial_NEIGHBOR')
m2_p = compute_metrics(transactions_p90cap, 'FibroticEpithelial_CENTER', 'Neutrophil_NEIGHBOR')
m3_p = compute_metrics(transactions_p90cap, 'Macrophage_CENTER',         'Macrophage_NEIGHBOR')

print('=' * 60)
print_metrics(m1_p, 'Neutrophil_CENTER', 'FibroticEpithelial_NEIGHBOR')
print()
print('=' * 60)
print_metrics(m2_p, 'FibroticEpithelial_CENTER', 'Neutrophil_NEIGHBOR')
print()
print('=' * 60)
print_metrics(m3_p, 'Macrophage_CENTER', 'Macrophage_NEIGHBOR')

Rule:  Neutrophil_CENTER
    ->  FibroticEpithelial_NEIGHBOR
  support(A)   = 0.2547  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 1.0685  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.1690  (avg min(w_A, w_B) per transaction; binary: fraction where both co-occur)
  confidence   = 0.663   (support(A,B) / support(A); binary: P(B present | A is CENTER) = 66.3%)
  lift         = 0.62   (confidence / support(B); >1 means co-occurrence stronger than chance)
  conviction   = -0.20  (how often A occurs without B being incorrectly predicted absent; >1 means rule is informative)

Rule:  FibroticEpithelial_CENTER
    ->  Neutrophil_NEIGHBOR
  support(A)   = 0.0916  (avg weight of A per transaction; binary: fraction of 1877 cells where A is CENTER)
  support(B)   = 3.8944  (avg weight of B per transaction; binary: fraction where B is NEIGHBOR)
  support(A,B) = 0.0811  (avg min(w_A, w_B) per transa

---
## Final Comparison

| | Binary | Normalized (bug) | Capped @1 | Raw Gaussian | Log Per-Type | P90 Cap Per-Type |
|---|---|---|---|---|---|---|
| support(CENTER) | true freq | **distorted** | true freq | true freq | true freq | true freq |
| support(NEIGHBOR) | ∈ [0,1] | ∈ [0,1] | ∈ [0,1] | ⚠️ can be > 1 | ∈ [0,1] | ⚠️ can be > 1 |
| Distance-sensitive? | No | Yes (distorted) | Yes | Yes | Yes | Yes |
| Support bias (rare in dense area)? | None | **Severe** | None | None | None | None |
| All metrics bounded? | Yes | Yes | Yes | ⚠️ No | Yes | ⚠️ No (abundant types) |
| Fair across cell types? | Yes | No | Yes | Yes | ⚠️ No | Yes — clipping is relative |
| Outlier suppression? | — | — | Global | None | Yes (log compress) | Yes (hard clip per type) |

**Binary** — interpretable ground truth.  
**Capped @1** — distance-sensitive, all metrics in [0,1], equal footing across cell types; recommended.  
**P90 Cap Per-Type** — removes top-10% extreme outliers per cell type while keeping raw scale; lift still valid; conviction/confidence break for abundant types.  
**Log Per-Type** — bounded to [0,1] but introduces abundance bias: rare cell types get disproportionate weight.  
**Raw** — most faithful; lift valid, conviction breaks when support > 1.  
**Normalized** — should be avoided: inflated lifts and biased support are pure artifacts.

In [93]:
# Side-by-side summary table
rows = []
for rule_name, mb, mn, mc, mr, ml, mp in [
    ('Neutrophil -> Fibrotic',   m1,  m1_n, m1_c, m1_r, m1_l, m1_p),
    ('Fibrotic -> Neutrophil',   m2,  m2_n, m2_c, m2_r, m2_l, m2_p),
    ('Macrophage -> Macrophage', m3,  m3_n, m3_c, m3_r, m3_l, m3_p),
]:
    rows.append({
        'Rule':           rule_name,
        'supp_A (Bin)':   f"{mb['supp_A']:.3f}",
        'supp_A (Norm)':  f"{mn['supp_A']:.3f}",
        'supp_A (Cap)':   f"{mc['supp_A']:.3f}",
        'supp_A (Raw)':   f"{mr['supp_A']:.3f}",
        'supp_A (Log)':   f"{ml['supp_A']:.3f}",
        'supp_A (P90)':   f"{mp['supp_A']:.3f}",
        'lift (Bin)':     f"{mb['lift']:.2f}",
        'lift (Norm)':    f"{mn['lift']:.2f}",
        'lift (Cap)':     f"{mc['lift']:.2f}",
        'lift (Raw)':     f"{mr['lift']:.2f}",
        'lift (Log)':     f"{ml['lift']:.2f}",
        'lift (P90)':     f"{mp['lift']:.2f}",
        'conf (Bin)':     f"{mb['confidence']:.2f}",
        'conf (Norm)':    f"{mn['confidence']:.2f}",
        'conf (Cap)':     f"{mc['confidence']:.2f}",
        'conf (Raw)':     f"{mr['confidence']:.2f}",
        'conf (Log)':     f"{ml['confidence']:.2f}",
        'conf (P90)':     f"{mp['confidence']:.2f}",
    })

df = pd.DataFrame(rows).set_index('Rule')
print(df.to_string())

                         supp_A (Bin) supp_A (Norm) supp_A (Cap) supp_A (Raw) supp_A (Log) supp_A (P90) lift (Bin) lift (Norm) lift (Cap) lift (Raw) lift (Log) lift (P90) conf (Bin) conf (Norm) conf (Cap) conf (Raw) conf (Log) conf (P90)
Rule                                                                                                                                                                                                                                         
Neutrophil -> Fibrotic          0.255         0.016        0.255        0.255        0.255        0.255       1.11        8.53       1.10       0.59       1.09       0.62       0.79        0.70       0.66       0.66       0.48       0.66
Fibrotic -> Neutrophil          0.092         0.007        0.092        0.092        0.092        0.092       1.24        3.46       1.27       0.21       1.09       0.23       0.94        0.85       0.89       0.89       0.48       0.89
Macrophage -> Macrophage        0.142         0.